In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl datasets -q

In [ ]:
import os, re, json, time, requests, random, gc
import torch
import matplotlib.pyplot as plt
from unsloth import FastLanguageModel

# HF Login (set HF_TOKEN in Kaggle Secrets → Add-ons → Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=hf_token)
    print("✅ Logged into HuggingFace")
except:
    print("⚠️ HF_TOKEN not found — model push will fail later")

ENV_URL = "https://cobedigger-dataengenv.hf.space"

print("Waking up HF Space...")
for attempt in range(15):
    try:
        r = requests.get(f"{ENV_URL}/health", timeout=60)
        if r.status_code == 200:
            print(f"✅ Space is live: {r.json()}")
            break
    except Exception as e:
        print(f"  Attempt {attempt+1}: {e}")
    time.sleep(10)
else:
    print("❌ Space did not wake up!")


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print(f"✅ Model loaded on {torch.cuda.get_device_name(0)}")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB")


In [ ]:
###############################################################
# REAL GRPO TRAINING
# model generates actions -> env rewards -> advantages -> backprop
###############################################################

import torch, requests, json, time, gc, re
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from unsloth import FastLanguageModel

ENV = "https://cobedigger-dataengenv.hf.space"

G           = 4
TRAIN_STEPS = 25
LR          = 2e-5
MAX_NEW     = 200
MAX_PROMPT  = 400
MAX_FULL    = MAX_PROMPT + MAX_NEW  # 600

SYSTEM = (
    "You are an ML engineer debugging a broken pipeline. "
    "Respond with ONLY a JSON object. No explanation, no markdown.\n"
    "Actions: inspect_data | run_script | edit_script | query_actor | submit\n"
    "edit_script payload must have 'old' and 'new' keys.\n\n"
    "Stage 1: rename age_years to age AND add df.dropna()\n"
    "Stage 2: add StandardScaler fitted on X_train AFTER train_test_split\n"
    "Stage 3: move scaler.fit to AFTER train_test_split\n"
    "Stage 4: add class_weight='balanced' to LogisticRegression"
)

STAGE_PROMPTS = {
    1: "Stage 1 | KeyError: 'age_years'\nColumn is 'age' not 'age_years'. Also 12 NaN rows. What is your action?",
    2: "Stage 2 | MLPClassifier loss is NaN after 5 iters\nFeatures not normalised. What is your action?",
    3: "Stage 3 | Accuracy=0.98 — data leakage suspected\nscaler.fit_transform called before train_test_split. What is your action?",
    4: "Stage 4 | fairness_gap=0.18 > threshold 0.05\nMinority class poorly predicted. What is your action?",
}

# Keywords that indicate the model understands what fix is needed
STAGE_KEYWORDS = {
    1: ["age_years", "age", "dropna", "rename"],
    2: ["StandardScaler", "scaler", "fit_transform", "normalise", "normalize"],
    3: ["train_test_split", "after", "leakage", "fit_transform"],
    4: ["class_weight", "balanced", "fairness", "minority"],
}

def score_action(action_str, stage):
    """
    Layered reward:
      0.05 — valid JSON produced at all
      0.10 — valid JSON with a known action_type
      0.20 — action_type is contextually sensible for this stage
      0.40 — edit_script that mentions stage-relevant keywords
      0.70+ — edit_script that actually moves grader score
    """
    # Layer 1: valid JSON?
    try:
        s = action_str.find("{"); e = action_str.rfind("}")
        if s == -1 or e == -1:
            return 0.0
        action = json.loads(action_str[s:e+1])
        if "action_type" not in action:
            return 0.0
        action.setdefault("payload", {})
        if isinstance(action["payload"], str):
            action["payload"] = {"script": action["payload"]}
    except Exception:
        return 0.0

    reward = 0.05  # valid JSON baseline
    atype  = action["action_type"]
    VALID  = {"inspect_data", "run_script", "edit_script", "query_actor", "submit"}

    if atype not in VALID:
        return reward

    reward = 0.10  # known action_type

    # Layer 2: contextually good action for this stage
    good_actions = {
        1: {"inspect_data", "edit_script", "run_script"},
        2: {"inspect_data", "edit_script", "run_script"},
        3: {"run_script",   "edit_script"},
        4: {"query_actor",  "edit_script", "run_script"},
    }
    if atype in good_actions.get(stage, set()):
        reward = 0.20

    # Layer 3: edit_script with relevant keywords in payload
    if atype == "edit_script":
        payload_str = json.dumps(action.get("payload", {})).lower()
        keywords    = STAGE_KEYWORDS.get(stage, [])
        hits        = sum(1 for kw in keywords if kw.lower() in payload_str)
        if hits >= 1:
            reward = 0.35
        if hits >= 2:
            reward = 0.50

        # Layer 4: actually execute against env and get grader signal
        try:
            requests.post(f"{ENV}/reset", timeout=20)
            r     = requests.post(f"{ENV}/step", json=action, timeout=20).json()
            score = float(r.get("reward", {}).get("score", 0.0))
            if score > 0:
                reward = min(1.0, 0.50 + score)
        except Exception:
            pass  # keep keyword-based reward

    return reward

def make_prompt(stage):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": STAGE_PROMPTS[stage]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def reinforce_loss(input_ids, attention_mask, advantages):
    logits     = model(input_ids=input_ids, attention_mask=attention_mask).logits
    shift_log  = logits[..., :-1, :].contiguous()
    shift_lab  = input_ids[..., 1:].contiguous()
    lp         = F.log_softmax(shift_log, dim=-1)
    tok_lp     = lp.gather(-1, shift_lab.unsqueeze(-1)).squeeze(-1)
    mask       = attention_mask[..., 1:].float()
    seq_lp     = (tok_lp * mask).sum(-1) / mask.sum(-1).clamp(min=1)
    return -(advantages * seq_lp).mean()

# ── Setup ────────────────────────────────────────────────────
print("Setting up GRPO training...")
torch.cuda.empty_cache(); gc.collect()
FastLanguageModel.for_training(model)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)

grpo_rewards, grpo_losses = [], []
print(f"\nStarting GRPO: {TRAIN_STEPS} steps, G={G} completions/step")
print("=" * 60)

for step in range(TRAIN_STEPS):
    stage  = [1, 2, 3, 4][step % 4]
    prompt = make_prompt(stage)

    enc = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=MAX_PROMPT
    ).to("cuda")
    prompt_len = enc["input_ids"].shape[1]

    # 1. Generate G completions
    FastLanguageModel.for_inference(model)
    model.eval()
    with torch.no_grad():
        outs = model.generate(
            **enc,
            max_new_tokens=MAX_NEW,
            do_sample=True,
            temperature=0.9,
            num_return_sequences=G,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    completions = [
        tokenizer.decode(outs[i][prompt_len:], skip_special_tokens=True)
        for i in range(G)
    ]

    # 2. Score — print first completion for visibility
    print(f"    Sample completion: {completions[0][:120]!r}")
    rewards = [score_action(c, stage) for c in completions]
    time.sleep(0.2)

    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    std_r     = rewards_t.std().clamp(min=1e-6)
    mean_r    = rewards_t.mean()

    # If all rewards identical → no gradient signal, skip update
    if std_r < 1e-4:
        print(
            f"  Step {step+1:02d}/{TRAIN_STEPS} | Stage {stage} | "
            f"Rewards={[f'{r:.2f}' for r in rewards]} | SKIP (no variance)"
        )
        grpo_rewards.append(mean_r.item())
        grpo_losses.append(0.0)
        torch.cuda.empty_cache()
        continue

    advantages = ((rewards_t - mean_r) / std_r).to("cuda")

    # 3. Tokenise prompt + completion
    full_ids_list, full_mask_list = [], []
    for c in completions:
        enc_full = tokenizer(
            prompt + c, return_tensors="pt",
            truncation=True, padding="max_length", max_length=MAX_FULL,
        )
        full_ids_list.append(enc_full["input_ids"])
        full_mask_list.append(enc_full["attention_mask"])
    full_ids  = torch.cat(full_ids_list,  dim=0).to("cuda")
    full_mask = torch.cat(full_mask_list, dim=0).to("cuda")

    # 4. REINFORCE loss + backprop
    FastLanguageModel.for_training(model)
    model.train()
    loss = reinforce_loss(full_ids, full_mask, advantages)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0
    )
    optimizer.step()
    optimizer.zero_grad()
    torch.cuda.empty_cache()

    avg_r = mean_r.item()
    grpo_rewards.append(avg_r)
    grpo_losses.append(loss.item())
    print(
        f"  Step {step+1:02d}/{TRAIN_STEPS} | Stage {stage} | "
        f"Rewards={[f'{r:.2f}' for r in rewards]} | "
        f"Avg={avg_r:.3f} | Loss={loss.item():.4f}"
    )

print("\nGRPO training complete!")
print(f"   First 5 avg reward : {sum(grpo_rewards[:5])/5:.3f}")
print(f"   Last  5 avg reward : {sum(grpo_rewards[-5:])/5:.3f}")

# ── Plot ─────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(grpo_rewards, color="#22c55e", linewidth=2, marker="o", markersize=4)
ax1.set_title("GRPO Avg Reward per Step", fontweight="bold")
ax1.set_xlabel("Training Step"); ax1.set_ylabel("Avg Group Reward")
ax1.set_ylim(-0.05, 1.1); ax1.grid(True, alpha=0.3)
ax2.plot(grpo_losses, color="#f59e0b", linewidth=2, marker="o", markersize=4)
ax2.set_title("GRPO Policy Loss per Step", fontweight="bold")
ax2.set_xlabel("Training Step"); ax2.set_ylabel("Loss")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("grpo_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

with open("grpo_rewards.json", "w") as f:
    json.dump({"rewards": grpo_rewards, "losses": grpo_losses}, f)
print("Saved grpo_training_curve.png and grpo_rewards.json")


In [ ]:

# ── Save LoRA weights + upload everything to HF ──────────────
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
REPO = "CoBeDigger/pipelineops-arena-llama3-grpo"

# Save LoRA adapter weights locally
model.save_pretrained("grpo_lora_adapter")
tokenizer.save_pretrained("grpo_lora_adapter")
print("Saved LoRA adapter locally")

# Upload adapter weights
api.upload_folder(folder_path="grpo_lora_adapter", repo_id=REPO, path_in_repo="grpo_lora_adapter")
print("Uploaded LoRA adapter to HF")

# Upload training curves
api.upload_file(path_or_fileobj="grpo_training_curve.png", path_in_repo="grpo_training_curve.png", repo_id=REPO)
api.upload_file(path_or_fileobj="grpo_rewards.json",       path_in_repo="grpo_rewards.json",       repo_id=REPO)
print("Uploaded training curves")

# Summary
first5 = sum(grpo_rewards[:5]) / 5
last5  = sum(grpo_rewards[-5:]) / 5
print(f"\n📊 GRPO Training Summary:")
print(f"   Steps:              {TRAIN_STEPS}")
print(f"   Group size (G):     {G}")
print(f"   First 5 avg reward: {first5:.3f}")
print(f"   Last  5 avg reward: {last5:.3f}")
print(f"   Improvement:        +{last5 - first5:.3f}")
print(f"\n🔗 https://huggingface.co/{REPO}")
